# Data Cleaning & Preprocessing

This notebook handles data cleaning, missing value imputation, outlier detection, and data normalization.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer, KNNImputer
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# Load dataset
df = pd.read_csv('../data/cleaned_weather_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


## 1. Missing Values Analysis


In [ ]:
# Check missing values
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing_data.index,
    'Missing Count': missing_data.values,
    'Missing Percentage': missing_percent.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Percentage', ascending=False)

print("Missing Values Summary:")
print(missing_df)

# Visualize missing values
if len(missing_df) > 0:
    plt.figure(figsize=(12, 8))
    sns.barplot(data=missing_df.head(20), x='Missing Percentage', y='Column')
    plt.title('Missing Values by Column')
    plt.xlabel('Missing Percentage (%)')
    plt.tight_layout()
    plt.savefig('../reports/missing_values.png', dpi=300, bbox_inches='tight')
    plt.show()


## 3. Outlier Detection
leave 

## 2. Handle Missing Values


In [ ]:
# Convert lastupdated to datetime
if 'lastupdated' in df.columns:
    df['lastupdated'] = pd.to_datetime(df['lastupdated'], errors='coerce')

# Separate numerical and categorical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove datetime columns from numerical
if 'lastupdated' in numerical_cols:
    numerical_cols.remove('lastupdated')

# Handle numerical missing values using KNN Imputation
if len(numerical_cols) > 0:
    knn_imputer = KNNImputer(n_neighbors=5)
    df[numerical_cols] = knn_imputer.fit_transform(df[numerical_cols])

# Handle categorical missing values using mode
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
        df[col].fillna(mode_value, inplace=True)

print("Missing values after imputation:", df.isnull().sum().sum())


In [ ]:
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

def detect_outliers_zscore(df, column, threshold=3):
    z_scores = np.abs((df[column] - df[column].mean()) / df[column].std())
    outliers = df[z_scores > threshold]
    return outliers

# Detect outliers for key numerical columns
key_columns = [col for col in ['temperature', 'humidity', 'precipitation', 'windspeed'] if col in df.columns]
outlier_summary = {}

for col in key_columns[:10]:  # Limit to first 10 to avoid too many columns
    try:
        outliers_iqr, _, _ = detect_outliers_iqr(df, col)
        outliers_zscore = detect_outliers_zscore(df, col)
        outlier_summary[col] = {
            'IQR Outliers': len(outliers_iqr),
            'Z-Score Outliers': len(outliers_zscore)
        }
    except:
        continue

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary).T
    print("Outlier Summary:")
    print(outlier_df)


## 4. Handle Outliers
it like 

In [ ]:
# Cap outliers using IQR method
for col in key_columns:
    try:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    except:
        continue

print("Outliers handled using IQR capping method")


## 5. Data Normalization


In [ ]:
# Standardize numerical features
scaler = StandardScaler()
df_scaled = df.copy()
if len(numerical_cols) > 0:
    df_scaled[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Save cleaned data
import os
os.makedirs('../data', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

df_scaled.to_csv('../data/cleaned_weather_data.csv', index=False)
df.to_csv('../data/weather_data_cleaned_no_scale.csv', index=False)

print("Cleaned data saved successfully!")
print(f"Final dataset shape: {df_scaled.shape}")
